# Jejak Bara v3.0 — notebook pemrosesan (data asli NASA)

Jalankan dari atas ke bawah. Hasil akhirnya: folder `keluaran/` (dibaca aplikasi web) dan `olahan/` (grafik untuk slide)
di Google Drive. Unduh `keluaran.zip` di sel terakhir, ekstrak menimpa `keluaran/` di repo, commit, push.

**Data NASA yang dipakai**
| Data | Koleksi | Untuk |
|---|---|---|
| NISAR L2 GCOV (HHHH, HVHV) | `NISAR_L2_GCOV_V1` → `_PROVISIONAL_V1` → `_BETA_V1` (ASF DAAC) | deteksi kehilangan tajuk, SLIC + Multi-Otsu |
| VIIRS 375 m active fire | FIRMS `VIIRS_SNPP_SP`, `VIIRS_SNPP_NRT`, `VIIRS_NOAA20_NRT` | label terkonfirmasi, titik bakar mulai, FRP |
| MERRA-2 aerosol | `M2TMNXAER` (GES DISC), `TOTEXTTAU` | muatan asap (AOD) per poligon |
| Citra optik berasap | NASA GIBS / Worldview (dipakai langsung di aplikasi) | konteks "smoke paradox" |

**Sebelum mulai:** unggah `jejak_bara.py` ke Drive `MyDrive/spaceapps_nisar/`. Login sekali ke ASF Vertex di browser untuk
menyetujui EULA NISAR (tanpa itu unduhan 401). Untuk MERRA-2, setujui aplikasi *NASA GESDISC DATA ARCHIVE* di profil Earthdata.

## 1. Pasang paket


In [ ]:
!pip install -q earthaccess rasterio rioxarray geopandas xarray h5netcdf h5py scikit-image netCDF4

## 2. Drive dan modul


In [ ]:
from google.colab import drive
import os, sys, json, glob
drive.mount('/content/drive')
K = '/content/drive/MyDrive/spaceapps_nisar'
for sub in ['mentah', 'olahan', 'keluaran']:
    os.makedirs(f'{K}/{sub}', exist_ok=True)
assert os.path.exists(f'{K}/jejak_bara.py'), 'Unggah jejak_bara.py ke ' + K + ' dulu'
sys.path.insert(0, K)
import importlib, jejak_bara as jb; importlib.reload(jb)
print('modul siap. AOI:', list(jb.AOI_KANDIDAT))


## 3. Login Earthdata
Prompt interaktif — password tidak tersimpan di notebook. Pastikan sudah login sekali ke ASF Vertex untuk menyetujui EULA.


In [ ]:
import earthaccess
auth = earthaccess.login(strategy='interactive', persist=True)
print('terautentikasi:', auth.authenticated)


## 4. AOI yang dipakai
Ubah `bbox` di sini kalau klaster titik panas (sel 6) ternyata di luar kotak. Kalteng dan Merauke: S dan N **negatif**.


In [ ]:
AOI = {k: dict(v) for k, v in jb.AOI_KANDIDAT.items()}
# contoh mempersempit: AOI['riau']['bbox'] = (101.8, 0.6, 102.6, 1.4)
for k, v in AOI.items(): print(f'{k:8s} {v["bbox"]}  {v["nama"]}')


## 5. Titik panas VIIRS (sekali untuk seluruh Indonesia)
Kunci gratis: `firms.modaps.eosdis.nasa.gov/api/map_key`. Dimasukkan lewat prompt, tidak disimpan di notebook.


In [ ]:
import io, requests, datetime as dt, pandas as pd, getpass
MAP_KEY = getpass.getpass('FIRMS MAP_KEY: ')
SUMBER = ['VIIRS_SNPP_SP', 'VIIRS_SNPP_NRT', 'VIIRS_NOAA20_NRT']   # sains dulu, lalu NRT
potongan, d = [], dt.date(2026, 6, 1)
while d < dt.date(2026, 11, 14):
    for s in SUMBER:
        r = requests.get(f'https://firms.modaps.eosdis.nasa.gov/api/country/csv/{MAP_KEY}/{s}/IDN/10/{d}', timeout=90)
        if r.ok and r.text.count('\n') > 1:
            df = pd.read_csv(io.StringIO(r.text)); df['sumber'] = s; potongan.append(df); break
    d += dt.timedelta(days=10)
hs = pd.concat(potongan, ignore_index=True).drop_duplicates(['latitude', 'longitude', 'acq_date', 'acq_time'])
HOTSPOT = f'{K}/mentah/hotspot_idn_2026.csv'
hs.to_csv(HOTSPOT, index=False)
print(len(hs), 'titik panas ->', HOTSPOT)
print('confidence:', hs['confidence'].value_counts().to_dict(), '(pipeline hanya memakai n dan h)')

## 6. Sebaran titik panas per AOI
Pastikan kotak AOI menangkap klaster terbesar. Kalau tidak, ubah sel 4 lalu jalankan ulang sel ini.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(AOI), figsize=(5 * len(AOI), 4.5))
for a, (k, v) in zip(ax, AOI.items()):
    w, s, e, n = v['bbox']
    sub = hs[(hs.longitude.between(w, e)) & (hs.latitude.between(s, n))]
    a.scatter(sub.longitude, sub.latitude, s=1, alpha=.3, c=pd.to_datetime(sub.acq_date).map(pd.Timestamp.toordinal), cmap='plasma')
    a.set_xlim(w, e); a.set_ylim(s, n); a.set_title(f'{k}: {len(sub):,} titik panas'); a.set_aspect('equal')
plt.tight_layout(); plt.show()
for k, v in AOI.items():
    w, s, e, n = v['bbox']; sub = hs[(hs.longitude.between(w, e)) & (hs.latitude.between(s, n))]
    if len(sub): print(k, 'puncak bulan:', pd.to_datetime(sub.acq_date).dt.to_period('M').value_counts().idxmax())


## 7. Peta gambut (KHG) dan batas kabupaten per AOI
Kalau salah satu sumber gagal diakses, unduh manual lalu simpan ke `mentah/<aoi>/khg.geojson` atau `kabupaten.geojson`.


In [ ]:
import geopandas as gpd
KHG_SVC = 'https://geoportal.menlhk.go.id/arcgis/rest/services/KLHK/Kesatuan_Hidrologis_Gambut_/MapServer'
info = requests.get(KHG_SVC, params={'f': 'json'}, timeout=60).json()
print('layer KHG:', [(l['id'], l['name']) for l in info.get('layers', [])])
KHG_LAYER = info['layers'][0]['id']   # ganti kalau layer poligon KHG bukan yang pertama

def ambil_khg(bbox):
    fitur, off = [], 0
    while True:
        r = requests.get(f'{KHG_SVC}/{KHG_LAYER}/query', timeout=120, params={
            'where': '1=1', 'outFields': '*', 'outSR': 4326, 'f': 'geojson',
            'geometry': ','.join(map(str, bbox)), 'geometryType': 'esriGeometryEnvelope',
            'inSR': 4326, 'spatialRel': 'esriSpatialRelIntersects', 'resultOffset': off}).json()
        fitur += r.get('features', [])
        if not r.get('exceededTransferLimit') and not r.get('properties', {}).get('exceededTransferLimit'): break
        off = len(fitur)
    return gpd.GeoDataFrame.from_features(fitur, crs='EPSG:4326')

gb = requests.get('https://www.geoboundaries.org/api/current/gbOpen/IDN/ADM2/', timeout=60).json()
KAB_ALL = gpd.read_file(gb['simplifiedGeometryGeoJSON'])
KOLOM_KAB = 'shapeName'

for k, v in AOI.items():
    os.makedirs(f'{K}/mentah/{k}', exist_ok=True)
    khg = ambil_khg(v['bbox']); khg.to_file(f'{K}/mentah/{k}/khg.geojson', driver='GeoJSON')
    kab = KAB_ALL.clip(v['bbox']); kab.to_file(f'{K}/mentah/{k}/kabupaten.geojson', driver='GeoJSON')
    print(f'{k}: {len(khg)} poligon KHG, {len(kab)} kabupaten')


## 7b. Muatan asap (AOD) dari MERRA-2 — opsional, tapi dibutuhkan untuk Pilar 4
Rata-rata bulanan `TOTEXTTAU` (AOD 550 nm) dari `M2TMNXAER` untuk bulan-bulan antara pra dan pasca. Grid 0,5° × 0,625°: kasar,
tapi reanalisis tidak pernah "buta asap" seperti MODIS/VIIRS optik. Kalau lambat, biarkan jalan sambil lanjut ke sel 8.

In [ ]:
TGL_ASAP = ('2026-07-01', '2026-10-31')       # sesuaikan dengan pra/pasca setelah sel 9
m2 = earthaccess.search_data(short_name='M2TMNXAER', temporal=TGL_ASAP)
f_m2 = earthaccess.download(m2, local_path='/content/merra2')
print(len(f_m2), 'berkas MERRA-2')
for k, v in AOI.items():
    jb.buat_aod_rerata(f_m2, v['bbox'], f'{K}/mentah/{k}/aod_merra2.tif', variabel='TOTEXTTAU')

## 8. Cari granule NISAR GCOV per AOI
Pilih **satu kelompok lintasan** (track + arah + frame yang sama) dengan tanggal terbanyak.
Membandingkan lintasan berbeda membuat perbedaan geometri terbaca sebagai perubahan palsu.


In [ ]:
import re
KOLEKSI = ['NISAR_L2_GCOV_V1', 'NISAR_L2_GCOV_PROVISIONAL_V1', 'NISAR_L2_GCOV_BETA_V1']

def daftar_granule(bbox, t0='2026-06-01', t1='2026-11-14'):
    for sn in KOLEKSI:
        hasil = earthaccess.search_data(short_name=sn, bounding_box=bbox, temporal=(t0, t1), count=500)
        if hasil: break
    rows = []
    for g in hasil:
        nama = g['umm']['GranuleUR']; b = nama.split('_')
        # NISAR_L2_PR_GCOV_<siklus>_<track>_<A/D>_<frame>_<mode>_<pol>_...
        url = next((u for u in g.data_links(access='external') if u.endswith('.h5')), None)
        tgl = re.search(r'_(\d{8})T\d{6}_', nama)
        rows.append(dict(tanggal=f'{tgl.group(1)[:4]}-{tgl.group(1)[4:6]}-{tgl.group(1)[6:]}' if tgl else None,
                         track=b[5] if len(b) > 9 else '?', arah=b[6] if len(b) > 9 else '?',
                         frame=b[7] if len(b) > 9 else '?', pol=b[9] if len(b) > 9 else '?',
                         koleksi=sn, nama=nama, url=url))
    return pd.DataFrame(rows, columns=['tanggal','track','arah','frame','pol','koleksi','nama','url']).sort_values('tanggal')

GRAN = {}
for k, v in AOI.items():
    df = daftar_granule(v['bbox']); GRAN[k] = df
    print(f'\n=== {k}: {len(df)} granule ({df.koleksi.iloc[0] if len(df) else "-"}) ===')
    if len(df): print(df.groupby(['track', 'arah', 'frame', 'pol']).tanggal.agg(['count', 'min', 'max']).sort_values('count', ascending=False).head(5))

## 9. Tetapkan kelompok lintasan dan tanggal
Default: kelompok dengan tanggal terbanyak, pra = tanggal paling awal, pasca = paling akhir.
Sesuaikan setelah melihat puncak titik panas di sel 6 — pra harus sebelum klaster muncul, pasca setelah mereda.
Polarisasi harus memuat HV (dual-pol `DH...` atau quad-pol `QP...`).


In [ ]:
PILIH = {}
for k, df in GRAN.items():
    df = df[df.pol.str.startswith(('DH', 'QP')) & df.url.notna()]
    if df.empty: print(k, ': tidak ada granule dual/quad-pol — AOI ini dilewati'); continue
    grp = df.groupby(['track', 'arah', 'frame']).size().idxmax()
    g = df[(df.track == grp[0]) & (df.arah == grp[1]) & (df.frame == grp[2])].drop_duplicates('tanggal')
    PILIH[k] = dict(granule=g, pra=g.tanggal.iloc[0], pasca=g.tanggal.iloc[-1])
    print(f'{k}: track {grp}, {len(g)} tanggal, pra {g.tanggal.iloc[0]}, pasca {g.tanggal.iloc[-1]}')

# contoh override: PILIH['riau']['pra'] = '2026-07-04'; PILIH['riau']['pasca'] = '2026-10-12'


## 10. Ambil potongan AOI (streaming — tidak mengunduh granule utuh)
Hanya bagian HDF5 yang beririsan AOI yang dibaca. Kalau streaming terlalu lambat atau gagal,
sel ini otomatis jatuh ke unduhan penuh di disk Colab sementara, memotong, lalu menghapus berkas besarnya.


In [ ]:
import shutil
for k, p in PILIH.items():
    out = f'{K}/mentah/{k}/gcov'; os.makedirs(out, exist_ok=True)
    for _, r in p['granule'].iterrows():
        if os.path.exists(f'{out}/{r.tanggal}_hv.tif'): continue
        print(f'{k} {r.tanggal}'); 
        try:
            fo = earthaccess.open([r.url])[0]
            jb.subset_gcov(fo, AOI[k]['bbox'], out, r.tanggal)
        except Exception as ex:
            print('  streaming gagal:', ex, '-> unduh penuh sementara')
            tmp = earthaccess.download([r.url], local_path='/content/tmp')[0]
            jb.subset_gcov(tmp, AOI[k]['bbox'], out, r.tanggal); os.remove(tmp)


## 11. Jalankan pipeline v3 per AOI
**Riau dulu sampai berhasil** (Checkpoint 1), lalu Kalteng, lalu Merauke (Checkpoint 2).

Ambang dipilih otomatis oleh Multi-Otsu pada superpixel SLIC di dalam KHG (dibatasi maksimal −1,8 dB).
Kalau histogram di sel 12 menunjukkan garis merah tidak di lembah, isi `AMBANG[k]` dengan angka (dB) — sebut di slide Validity.
`RESOLUSI` 80 m untuk percobaan pertama (cepat), 40 m kalau waktu cukup. Jangan menimpa `keluaran/` yang bagus dengan run gagal.

In [ ]:
AMBANG = {'riau': None, 'kalteng': None, 'merauke': None}   # None = Multi-Otsu otomatis
RESOLUSI = 80
JALANKAN = ['riau']          # tambah 'kalteng', 'merauke' setelah Riau beres

for k in JALANKAN:
    p = PILIH[k]; g = f'{K}/mentah/{k}/gcov'
    deret = [(t, f'{g}/{t}_hv.tif', f'{g}/{t}_hh.tif') for t in p['granule'].tanggal if os.path.exists(f'{g}/{t}_hv.tif')]
    row = lambda t: p['granule'][p['granule'].tanggal == t].iloc[0]
    aod = f'{K}/mentah/{k}/aod_merra2.tif'
    jb.proses(
        aoi_nama=AOI[k]['nama'], aoi_bbox=AOI[k]['bbox'],
        tanggal_pra=p['pra'], tanggal_pasca=p['pasca'],
        hv_pra=f'{g}/{p["pra"]}_hv.tif', hv_pasca=f'{g}/{p["pasca"]}_hv.tif',
        hh_pra=f'{g}/{p["pra"]}_hh.tif', hh_pasca=f'{g}/{p["pasca"]}_hh.tif',
        khg=f'{K}/mentah/{k}/khg.geojson', hotspot_csv=HOTSPOT,
        kabupaten=f'{K}/mentah/{k}/kabupaten.geojson', kolom_kabupaten=KOLOM_KAB,
        deret=deret, aod_tif=aod if os.path.exists(aod) else None, sumber_aod='MERRA-2 M2TMNXAER TOTEXTTAU',
        granule_pra=row(p['pra'])['nama'], granule_pasca=row(p['pasca'])['nama'],
        produk=row(p['pra'])['koleksi'], mode='nisar',
        folder_keluaran=f'{K}/keluaran/{k}', folder_olahan=f'{K}/olahan/{k}',
        param=jb.Parameter(ambang_hv_db=AMBANG[k], resolusi_m=RESOLUSI, ukuran_filter=3 if RESOLUSI >= 80 else 5),
    )

## 12. Periksa histogram Multi-Otsu dan plot asap–hotspot (keduanya untuk slide)

In [ ]:
from IPython.display import Image, display
for k in JALANKAN:
    print(k); display(Image(f'{K}/olahan/{k}/histogram_multiotsu.png')); display(Image(f'{K}/olahan/{k}/plot_asap_hotspot.png'))
    st = json.load(open(f'{K}/keluaran/{k}/statistik.json'))
    print({x: st[x] for x in ['total_ha','terkonfirmasi_ha','kandidat_tersembunyi_ha','perlu_verifikasi_ha',
                             'rasio_kebutaan_termal_akibat_asap_persen','arah_rambatan_dampak_dominan','ambang_dipakai_db']})

## 13. Tulis indeks dan bungkus untuk repo


In [ ]:
jb.tulis_indeks(f'{K}/keluaran')
assert all(a['mode'] == 'nisar' for a in json.load(open(f'{K}/keluaran/aoi.json'))['aoi'] if a['slug'] in JALANKAN)
shutil.make_archive('/content/keluaran', 'zip', f'{K}/keluaran')
shutil.make_archive('/content/olahan', 'zip', f'{K}/olahan')
from google.colab import files; files.download('/content/keluaran.zip'); files.download('/content/olahan.zip')
print('Ekstrak keluaran.zip menimpa folder keluaran/ di repo, lalu commit + push. olahan.zip untuk slide.')

---
**Periksa di QGIS sebelum commit:** buka `keluaran/<aoi>/terbakar.geojson` di atas `mentah/<aoi>/gcov/<pasca>_hv.tif`.
Poligon harus menempel di area gelap, bukan bergeser, dan tidak ada poligon di luar garis KHG.
